# STCP - Algoritmo de Decisão de Carregamento

Sistema que responde em tempo real à pergunta: **qual o autocarro que deve carregar a seguir, em que estação, e com que urgência?**

O algoritmo executa 7 fases sequenciais:

- **Fase 1** — Recolher estado actual de todos os dispositivos
- **Fase 2** — Análise energética de cada autocarro
- **Fase 3** — Verificar elegibilidade (restrições de segurança)
- **Fase 4** — Calcular indicador de pressão da frota
- **Fase 5** — Calcular score de urgência
- **Fase 6** — Alocar estações e ajustar potência
- **Fase 7** — Gerar decisões finais

**Versão:** 2.0 — Outubro 2026  
**Autor:** Luís Pereira — Faculdade de Ciências Empresariais do Porto

## Configuração — Parâmetros e Constantes

Todos os parâmetros baseados nas especificações do CaetanoBus e.City Gold e na literatura académica (He et al. 2023, Liu et al. 2022).

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import math

# --- PARÂMETROS DO AUTOCARRO (CaetanoBus e.City Gold) ---
BATERIA_CAPACIDADE_KWH   = 350.0   # Capacidade total da bateria (kWh)
CONSUMO_POR_KM           = 1.2     # Consumo médio (kWh/km)
SOC_MIN                  = 20.0    # Nível mínimo de segurança (%)
SOC_MAX                  = 95.0    # Nível máximo de carregamento (%)
SOC_IDEAL_MIN            = 20.0    # Início da faixa operacional ideal (%)
SOC_IDEAL_MAX            = 80.0    # Fim da faixa operacional ideal (%)
MARGEM_SEGURANCA         = 1.20    # Margem de segurança no regresso (20% extra)
TEMP_BATERIA_MAX         = 45.0    # Temperatura máxima da bateria (°C)
TEMP_BATERIA_IDEAL_MIN   = 15.0    # Temperatura ideal mínima (°C)
TEMP_BATERIA_IDEAL_MAX   = 35.0    # Temperatura ideal máxima (°C)

# --- PARÂMETROS DAS ESTAÇÕES ---
POTENCIA_MAX_ESTACAO     = 120.0   # Potência máxima por estação (kW)
POTENCIA_MAX_REDE        = 360.0   # Limite total da rede eléctrica (kW) — 3 × 120 kW
EFICIENCIA_CARREGAMENTO  = 0.92    # Eficiência do carregamento
TEMP_CARREGADOR_MAX      = 60.0    # Temperatura máxima do carregador (°C)

# --- LOCALIZAÇÃO DO DEPÓSITO (Via Norte / Bonjóia, Porto) ---
DEPOT_LAT = 41.152661
DEPOT_LON = -8.579658

# Distâncias e paragens baseadas nos dados reais da STCP (API pública stcp.pt),
# seguidas pelo simulador via interpolação de GPS (ROUTE_STOPS em electric_bus_simulator.py).
# --- DISTÂNCIAS TOTAIS DAS ROTAS (metros, dados reais da STCP) ---
DIST_LINHA_200 = 8640.2    # BUS-001: Bolhão - Castelo do Queijo (30 paragens)
DIST_LINHA_201 = 8389.7    # BUS-002: Aliados - Viso (26 paragens)
DIST_LINHA_202 = 9482.3    # BUS-003: Aliados - Passeio Alegre (30 paragens)

print("=" * 70)
print("CONFIGURAÇÃO CARREGADA")
print("=" * 70)
print(f"Capacidade da bateria:     {BATERIA_CAPACIDADE_KWH} kWh")
print(f"Consumo médio:             {CONSUMO_POR_KM} kWh/km")
print(f"Faixa operacional ideal:   {SOC_IDEAL_MIN}% - {SOC_IDEAL_MAX}%")
print(f"Potência máxima estação:   {POTENCIA_MAX_ESTACAO} kW")
print(f"Limite total da rede:      {POTENCIA_MAX_REDE} kW")
print(f"Depósito:                  {DEPOT_LAT}, {DEPOT_LON}")
print("=" * 70)

CONFIGURAÇÃO CARREGADA
Capacidade da bateria:     350.0 kWh
Consumo médio:             1.2 kWh/km
Faixa operacional ideal:   20.0% - 80.0%
Potência máxima estação:   120.0 kW
Limite total da rede:      360.0 kW
Depósito:                  41.152661, -8.579658


## Fase 1 — Estado Actual de Todos os Dispositivos

Recolher o registo mais recente de cada autocarro e de cada estação.
Usa `ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC)` para garantir que apenas o registo mais recente de cada dispositivo é considerado.

Também é criada a tabela de configuração das rotas que será usada nas fases seguintes.

In [0]:
print("=" * 70)
print("FASE 1 — ESTADO ACTUAL")
print("=" * 70)

# Criar tabela de configuração das rotas, com as distâncias e paragens reais da
# STCP (API pública stcp.pt): 30/26/30 paragens.
spark.sql("DROP TABLE IF EXISTS route_config")
spark.createDataFrame([
    ("BUS-001", "200", "Bolhao - Castelo do Queijo", "Alta",    8640.2, 30),
    ("BUS-002", "201", "Aliados - Viso",             "Normal",  8389.7, 26),
    ("BUS-003", "202", "Aliados - Passeio Alegre",   "Normal",  9482.3, 30)
], ["device_id", "linha", "nome_rota", "prioridade", "distancia_total_m", "num_paragens"]
).write.mode("overwrite").saveAsTable("route_config")

# Estado actual dos autocarros
print("\n--- AUTOCARROS ---")
display(spark.sql("""
    SELECT
        b.device_id,
        ROUND(b.batteryLevel, 1)       AS bateria_pct,
        ROUND(b.batteryKwh, 1)         AS bateria_kwh,
        ROUND(b.batteryTemperature, 1) AS temp_bateria_C,
        b.state                        AS estado,
        ROUND(b.distanceTraveled, 0)   AS dist_percorrida_m,
        ROUND(b.latitude, 6)           AS lat,
        ROUND(b.longitude, 6)          AS lon,
        r.nome_rota,
        r.prioridade
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) AS rn
        FROM bus_telemetry_raw
    ) b
    JOIN route_config r ON b.device_id = r.device_id
    WHERE b.rn = 1
    ORDER BY b.device_id
"""))

# Estado actual das estações
print("\n--- ESTAÇÕES DE CARREGAMENTO ---")
display(spark.sql("""
    SELECT
        device_id,
        state                          AS estado,
        ROUND(currentPower, 1)         AS potencia_kw,
        ROUND(chargerTemperature, 1)   AS temp_carregador_C,
        ROUND(currentEfficiency, 1)    AS eficiencia_pct,
        ROUND(energyDelivered, 1)      AS energia_entregue_kwh,
        connectedBus                   AS autocarro_ligado
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) AS rn
        FROM charger_telemetry_raw
    )
    WHERE rn = 1
    ORDER BY device_id
"""))

# Estado agregado da frota
print("\n--- ESTADO AGREGADO DA FROTA ---")
display(spark.sql("""
    SELECT
        COUNT(*)                                                    AS total_autocarros,
        SUM(CASE WHEN state = 'inTransit' THEN 1 ELSE 0 END)       AS em_transito,
        SUM(CASE WHEN state = 'parked'    THEN 1 ELSE 0 END)       AS estacionados,
        SUM(CASE WHEN state = 'charging'  THEN 1 ELSE 0 END)       AS a_carregar
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) AS rn
        FROM bus_telemetry_raw
    )
    WHERE rn = 1
"""))

# Estado agregado das estações
print("\n--- ESTADO AGREGADO DAS ESTAÇÕES ---")
display(spark.sql(f"""
    SELECT
        COUNT(*)                                                                              AS total_estacoes,
        SUM(CASE WHEN state = 'available' AND chargerTemperature < {TEMP_CARREGADOR_MAX} THEN 1 ELSE 0 END) AS disponiveis,
        SUM(CASE WHEN state = 'occupied'  THEN 1 ELSE 0 END)                                 AS ocupadas,
        SUM(CASE WHEN state = 'fault'     THEN 1 ELSE 0 END)                                 AS em_avaria,
        ROUND(SUM(currentPower), 1)                                                           AS potencia_total_kw,
        ROUND({POTENCIA_MAX_REDE} - SUM(currentPower), 1)                                    AS capacidade_disponivel_kw
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) AS rn
        FROM charger_telemetry_raw
    )
    WHERE rn = 1
"""))

FASE 1 — ESTADO ACTUAL

--- AUTOCARROS ---


device_id,bateria_pct,bateria_kwh,temp_bateria_C,estado,dist_percorrida_m,lat,lon,nome_rota,prioridade
BUS-001,17.8,62.1,23.2,headingToCharging,8190.0,41.148164,-8.608651,Bolhao - Castelo do Queijo,Alta
BUS-002,45.9,160.7,26.0,charging,10418.0,41.152661,-8.579658,Aliados - Viso,Normal
BUS-003,81.2,284.2,26.3,inTransit,0.0,41.152661,-8.579658,Aliados - Passeio Alegre,Normal



--- ESTAÇÕES DE CARREGAMENTO ---


device_id,estado,potencia_kw,temp_carregador_C,eficiencia_pct,energia_entregue_kwh,autocarro_ligado
CS-001,occupied,120.0,45.0,92.1,35.9,BUS-002
CS-002,fault,0.0,28.7,0.0,0.0,null
CS-003,occupied,120.0,35.7,91.2,23.2,BUS-003



--- ESTADO AGREGADO DA FROTA ---


total_autocarros,em_transito,estacionados,a_carregar
3,1,0,1



--- ESTADO AGREGADO DAS ESTAÇÕES ---


total_estacoes,disponiveis,ocupadas,em_avaria,potencia_total_kw,capacidade_disponivel_kw
3,0,2,1,240.0,120.0


## Fase 2 — Análise Energética

Para cada autocarro, calcular:

**Distância ao depósito** usando a aproximação plana da fórmula de Haversine (erro < 0.5% para distâncias < 50 km):
```
D_depot = 111.0 × sqrt((lat_depot - lat)² + (lon_depot - lon)² × cos(lat × π/180)²)
```

**Distância restante na rota**:
```
D_rota = distancia_total_rota - distanceTraveled
```

**Bateria necessária para completar a rota** (em %):
```
SOC_rota = (D_rota / 1000 × consumo_km / capacidade_bat) × 100
```

**Bateria necessária para regressar ao depósito** com margem de 20%:
```
SOC_volta = (D_depot × consumo_km × 1.20 / capacidade_bat) × 100
```

**Bateria total necessária**:
```
SOC_necesario = SOC_rota + SOC_volta + SOC_min (20%)
```

**Défice de bateria**:
```
delta_SOC = SOC_necesario - SOC_actual
  Se delta_SOC > 0 → autocarro em défice, deve carregar
  Se delta_SOC ≤ 0 → autocarro com reserva suficiente
```

**Nível previsto à chegada ao depósito** (sem carregamento):
```
SOC_previsto = SOC_actual - SOC_rota - SOC_volta
  Se SOC_previsto < SOC_min → não consegue completar o percurso em segurança
```

In [0]:
print("=" * 70)
print("FASE 2 — ANÁLISE ENERGÉTICA")
print("=" * 70)

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW analise_energetica AS
    WITH latest_bus AS (
        SELECT
            b.device_id,
            ROUND(b.batteryLevel, 2)       AS soc_actual,
            ROUND(b.batteryKwh, 2)         AS bateria_kwh,
            ROUND(b.batteryTemperature, 2) AS temp_bateria,
            b.state,
            b.latitude,
            b.longitude,
            b.distanceTraveled             AS dist_percorrida_m,
            r.nome_rota,
            r.prioridade,
            r.distancia_total_m,
            r.num_paragens
        FROM (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) AS rn
            FROM bus_telemetry_raw
        ) b
        JOIN route_config r ON b.device_id = r.device_id
        WHERE b.rn = 1
    ),
    com_distancias AS (
        SELECT
            *,
            -- Distância ao depósito (km) - aproximação plana de Haversine
            ROUND(
                111.0 * SQRT(
                    POW({DEPOT_LAT} - latitude, 2) +
                    POW(({DEPOT_LON} - longitude) * COS(RADIANS(latitude)), 2)
                ), 3
            ) AS dist_depot_km,
            -- Distância restante na rota (metros)
            GREATEST(distancia_total_m - dist_percorrida_m, 0) AS dist_rota_restante_m
        FROM latest_bus
    ),
    com_energia AS (
        SELECT
            *,
            -- SOC necessário para completar a rota actual (%)
            ROUND(
                (dist_rota_restante_m / 1000.0 * {CONSUMO_POR_KM} / {BATERIA_CAPACIDADE_KWH}) * 100,
                2
            ) AS soc_rota_pct,
            -- SOC necessário para regressar ao depósito com margem de 20% (%)
            ROUND(
                (dist_depot_km * {CONSUMO_POR_KM} * {MARGEM_SEGURANCA} / {BATERIA_CAPACIDADE_KWH}) * 100,
                2
            ) AS soc_volta_pct
        FROM com_distancias
    )
    SELECT
        device_id,
        nome_rota,
        prioridade,
        state,
        temp_bateria,
        soc_actual,
        bateria_kwh,
        dist_depot_km,
        dist_rota_restante_m,
        soc_rota_pct,
        soc_volta_pct,
        -- Bateria total necessária = rota + volta + mínimo de segurança
        ROUND(soc_rota_pct + soc_volta_pct + {SOC_MIN}, 2)                AS soc_necesario_pct,
        -- Défice de bateria (positivo = em défice, deve carregar)
        ROUND((soc_rota_pct + soc_volta_pct + {SOC_MIN}) - soc_actual, 2) AS delta_soc,
        -- Nível previsto à chegada ao depósito sem carregamento
        ROUND(soc_actual - soc_rota_pct - soc_volta_pct, 2)               AS soc_previsto_chegada,
        -- Paragem actual (baseada em distanceTraveled vs route_stops)
        -- Nota: requer JOIN com tabela route_stops para paragem exacta
        ROUND(dist_percorrida_m / NULLIF(distancia_total_m, 0) * num_paragens, 0) AS paragem_estimada,
        GREATEST(num_paragens - ROUND(dist_percorrida_m / NULLIF(distancia_total_m, 0) * num_paragens, 0), 0) AS paragens_restantes
    FROM com_energia
""")

display(spark.sql("""
    SELECT
        device_id,
        nome_rota,
        prioridade,
        soc_actual         AS bateria_actual_pct,
        soc_rota_pct       AS necessario_rota_pct,
        soc_volta_pct      AS necessario_volta_pct,
        soc_necesario_pct  AS total_necessario_pct,
        delta_soc          AS deficit_pct,
        soc_previsto_chegada AS previsto_deposito_pct,
        dist_depot_km,
        paragens_restantes
    FROM analise_energetica
    ORDER BY delta_soc DESC
"""))

FASE 2 — ANÁLISE ENERGÉTICA


device_id,nome_rota,prioridade,bateria_actual_pct,necessario_rota_pct,necessario_volta_pct,total_necessario_pct,deficit_pct,previsto_deposito_pct,dist_depot_km,paragens_restantes
BUS-001,Bolhao - Castelo do Queijo,Alta,17.75,0.15,1.02,21.17,3.42,16.58,2.474,2.0
BUS-002,Aliados - Viso,Normal,45.92,0.0,0.0,20.0,-25.92,45.92,0.0,0.0
BUS-003,Aliados - Passeio Alegre,Normal,81.21,3.25,0.0,23.25,-57.96,77.96,0.0,30.0


## Fase 3 — Elegibilidade e Restrições de Segurança

Um autocarro é **elegível** para carregamento se e só se satisfizer as três condições:

- **R1** — Temperatura da bateria ≤ 45°C (acima deste valor há risco de degradação ou incêndio)
- **R2** — SOC actual < 95% (se já estiver acima do máximo, não é necessário carregar)
- **R3** — delta_SOC > 0 (existe défice real de bateria)

Uma estação é **disponível** se:
- Estado = `available`
- Temperatura do carregador < 60°C

In [0]:
print("=" * 70)
print("FASE 3 — ELEGIBILIDADE E RESTRIÇÕES DE SEGURANÇA")
print("=" * 70)

# Elegibilidade dos autocarros
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW autocarros_elegiveis AS
    SELECT
        *,
        -- Verificar as 3 restrições
        CASE WHEN temp_bateria <= {TEMP_BATERIA_MAX} THEN 1 ELSE 0 END  AS r1_temperatura_ok,
        CASE WHEN soc_actual < {SOC_MAX}             THEN 1 ELSE 0 END  AS r2_nivel_ok,
        CASE WHEN delta_soc > 0                      THEN 1 ELSE 0 END  AS r3_deficit_ok,
        -- Elegível = satisfaz as 3 restrições
        CASE
            WHEN temp_bateria <= {TEMP_BATERIA_MAX}
             AND soc_actual < {SOC_MAX}
             AND delta_soc > 0
            THEN 1 ELSE 0
        END AS elegivel,
        -- Razão da não elegibilidade
        CASE
            WHEN temp_bateria > {TEMP_BATERIA_MAX} THEN 'BLOQUEADO - Temperatura da bateria acima de {TEMP_BATERIA_MAX}C'
            WHEN soc_actual >= {SOC_MAX}           THEN 'BATERIA CHEIA - SOC acima de {SOC_MAX}%'
            WHEN delta_soc <= 0                    THEN 'SEM NECESSIDADE - Bateria suficiente para completar o percurso'
            ELSE 'ELEGIVEL'
        END AS razao_elegibilidade
    FROM analise_energetica
""")

# Disponibilidade das estações
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW estacoes_estado AS
    SELECT
        device_id,
        state,
        ROUND(currentPower, 1)       AS potencia_kw,
        ROUND(chargerTemperature, 1) AS temp_carregador,
        connectedBus,
        ROUND(energyDelivered, 1)    AS energia_entregue_kwh,
        CASE
            WHEN state = 'available' AND chargerTemperature < {TEMP_CARREGADOR_MAX} THEN 1
            ELSE 0
        END AS disponivel,
        CASE
            WHEN state = 'available' AND chargerTemperature < {TEMP_CARREGADOR_MAX} THEN 'DISPONIVEL'
            WHEN state = 'occupied'                                                  THEN 'OCUPADA'
            WHEN state = 'fault'                                                     THEN 'AVARIA'
            WHEN chargerTemperature >= {TEMP_CARREGADOR_MAX}                        THEN 'SOBREAQUECIDA'
            ELSE 'INDISPONIVEL'
        END AS classificacao
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) AS rn
        FROM charger_telemetry_raw
    )
    WHERE rn = 1
""")

print("\n--- ELEGIBILIDADE DOS AUTOCARROS ---")
display(spark.sql("""
    SELECT device_id, nome_rota, soc_actual, temp_bateria, delta_soc,
           elegivel, razao_elegibilidade
    FROM autocarros_elegiveis
    ORDER BY elegivel DESC, delta_soc DESC
"""))

print("\n--- DISPONIBILIDADE DAS ESTAÇÕES ---")
display(spark.sql("""
    SELECT device_id, state, potencia_kw, temp_carregador,
           connectedBus, classificacao
    FROM estacoes_estado
    ORDER BY disponivel DESC, device_id
"""))

FASE 3 — ELEGIBILIDADE E RESTRIÇÕES DE SEGURANÇA

--- ELEGIBILIDADE DOS AUTOCARROS ---


device_id,nome_rota,soc_actual,temp_bateria,delta_soc,elegivel,razao_elegibilidade
BUS-001,Bolhao - Castelo do Queijo,17.75,23.17,3.42,1,ELEGIVEL
BUS-002,Aliados - Viso,45.92,25.98,-25.92,0,SEM NECESSIDADE - Bateria suficiente para completar o percurso
BUS-003,Aliados - Passeio Alegre,81.21,26.34,-57.96,0,SEM NECESSIDADE - Bateria suficiente para completar o percurso



--- DISPONIBILIDADE DAS ESTAÇÕES ---


device_id,state,potencia_kw,temp_carregador,connectedBus,classificacao
CS-001,occupied,120.0,45.0,BUS-002,OCUPADA
CS-002,fault,0.0,28.7,null,AVARIA
CS-003,occupied,120.0,35.7,BUS-003,OCUPADA


## Fase 4 — Indicador de Pressão da Frota

O indicador de pressão Π mede a relação entre a procura de carregamento e a capacidade disponível:

```
N_deficit  = número de autocarros com delta_SOC > 0
N_cs_avail = número de estações disponíveis
Π = N_deficit / max(N_cs_avail, 1)
```

Interpretação:
- **Π ≤ 1** — Operação normal, estações suficientes para todos
- **1 < Π ≤ 2** — Pressão moderada, priorização necessária
- **Π > 2** — Pressão crítica, escassez severa de estações

In [0]:
print("=" * 70)
print("FASE 4 — INDICADOR DE PRESSÃO DA FROTA")
print("=" * 70)

pressao = spark.sql("""
    SELECT
        (SELECT COUNT(*) FROM autocarros_elegiveis WHERE delta_soc > 0)    AS n_deficit,
        (SELECT COUNT(*) FROM estacoes_estado WHERE disponivel = 1)         AS n_estacoes_disponiveis,
        (SELECT SUM(potencia_kw) FROM estacoes_estado)                      AS potencia_rede_actual_kw,
        ROUND(
            (SELECT COUNT(*) FROM autocarros_elegiveis WHERE delta_soc > 0) /
            GREATEST((SELECT COUNT(*) FROM estacoes_estado WHERE disponivel = 1), 1),
        2) AS indicador_pressao
""").collect()[0]

PI = pressao['indicador_pressao']
P_REDE_ACTUAL = pressao['potencia_rede_actual_kw'] or 0

print(f"Autocarros em défice:       {pressao['n_deficit']}")
print(f"Estações disponíveis:       {pressao['n_estacoes_disponiveis']}")
print(f"Potência actual na rede:    {P_REDE_ACTUAL} kW")
print(f"Capacidade disponível:      {POTENCIA_MAX_REDE - P_REDE_ACTUAL} kW")
print(f"Indicador de pressão (Π):   {PI}")

if PI <= 1:
    print(f"\nMODO: OPERAÇÃO NORMAL — Estações suficientes para todos os autocarros em défice")
elif PI <= 2:
    print(f"\nMODO: PRESSÃO MODERADA — Mais autocarros em défice do que estações disponíveis")
else:
    print(f"\nMODO: PRESSÃO CRÍTICA — Escassez severa de estações")

FASE 4 — INDICADOR DE PRESSÃO DA FROTA
Autocarros em défice:       1
Estações disponíveis:       0
Potência actual na rede:    240.0 kW
Capacidade disponível:      120.0 kW
Indicador de pressão (Π):   1.0

MODO: OPERAÇÃO NORMAL — Estações suficientes para todos os autocarros em défice


## Fase 5 — Score de Urgência

O score de urgência U determina a ordem de alocação. É composto por 4 componentes:

**P_serv — Prioridade do serviço:**
```
Crítica = 40 pts | Alta = 30 pts | Normal = 20 pts
```

**P_bat — Défice de bateria:**
```
delta_SOC > 0          → 60 pts  (em défice)
delta_SOC > -10%       → 30 pts  (margem reduzida)
caso contrário         →  0 pts
```

**P_dist — Risco de não regressar ao depósito:**
```
SOC_actual < SOC_volta            → 100 pts  (não consegue regressar — emergência)
SOC_actual < SOC_volta + 5%       →  60 pts  (margem crítica)
caso contrário                    →   0 pts
```

**P_frota — Pressão da frota** (só activo quando Π > 1):
```
Π > 1 e estado = inTransit  → 20 pts
Π > 1 e estado = parked     → 10 pts
Π ≤ 1                       →  0 pts
```

**Score total:**
```
U = P_serv + P_bat + P_dist + P_frota
Intervalo teórico: 20 a 220 pontos
```

In [0]:
print("=" * 70)
print("FASE 5 — SCORE DE URGÊNCIA")
print("=" * 70)

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW scores_urgencia AS
    SELECT
        device_id,
        nome_rota,
        prioridade,
        state,
        soc_actual,
        delta_soc,
        soc_volta_pct,
        soc_necesario_pct,
        soc_previsto_chegada,
        temp_bateria,
        dist_depot_km,
        paragens_restantes,
        elegivel,
        razao_elegibilidade,

        -- P_serv: Prioridade do serviço
        CASE prioridade
            WHEN 'Critica' THEN 40
            WHEN 'Alta'    THEN 30
            ELSE                20
        END AS p_serv,

        -- P_bat: Gravidade do défice de bateria
        CASE
            WHEN delta_soc > 0   THEN 60
            WHEN delta_soc > -10 THEN 30
            ELSE                      0
        END AS p_bat,

        -- P_dist: Risco de não regressar ao depósito
        CASE
            WHEN soc_actual < soc_volta_pct          THEN 100
            WHEN soc_actual < soc_volta_pct + 5      THEN 60
            ELSE                                          0
        END AS p_dist,

        -- P_frota: Pressão da frota (só activo quando Π > 1)
        CASE
            WHEN {PI} > 1 AND state = 'inTransit' THEN 20
            WHEN {PI} > 1 AND state = 'parked'    THEN 10
            ELSE                                       0
        END AS p_frota

    FROM autocarros_elegiveis
""")

# Adicionar score total e ordenar
spark.sql("""
    CREATE OR REPLACE TEMP VIEW scores_ordenados AS
    SELECT
        *,
        (p_serv + p_bat + p_dist + p_frota) AS score_total
    FROM scores_urgencia
    ORDER BY elegivel DESC, score_total DESC
""")

display(spark.sql("""
    SELECT
        device_id,
        nome_rota,
        soc_actual,
        delta_soc,
        p_serv,
        p_bat,
        p_dist,
        p_frota,
        score_total,
        elegivel,
        razao_elegibilidade
    FROM scores_ordenados
    ORDER BY elegivel DESC, score_total DESC
"""))

FASE 5 — SCORE DE URGÊNCIA


device_id,nome_rota,soc_actual,delta_soc,p_serv,p_bat,p_dist,p_frota,score_total,elegivel,razao_elegibilidade
BUS-001,Bolhao - Castelo do Queijo,17.75,3.42,30,60,0,0,90,1,ELEGIVEL
BUS-002,Aliados - Viso,45.92,-25.92,20,0,0,0,20,0,SEM NECESSIDADE - Bateria suficiente para completar o percurso
BUS-003,Aliados - Passeio Alegre,81.21,-57.96,20,0,0,0,20,0,SEM NECESSIDADE - Bateria suficiente para completar o percurso


## Fase 6 — Alocação de Estações e Ajuste de Potência

**Alocação:** Os autocarros elegíveis são atendidos por ordem decrescente de score. Cada um recebe a primeira estação disponível.

**Potência final** ajustada por 3 factores em cascata:

```
P_temp = ajuste por temperatura da bateria:
  T > 45°C          →   0 kW  (bloqueado)
  35°C < T ≤ 45°C   →  80 kW  (reduzido)
  T < 15°C           →  50 kW  (pré-aquecimento)
  15°C ≤ T ≤ 35°C   → 120 kW  (normal)

f_bal = factor de balanceamento da rede:
  P_rede_actual > 90% de 360 kW  → 0.80
  P_rede_actual > 80% de 360 kW  → 0.90
  caso contrário                  → 1.00

f_urgencia = factor de urgência relativa (só quando Π > 1):
  score ≥ 150  → 1.00
  score ≥ 100  → 0.90
  score < 100  → 0.80

P_final = min(P_max, P_disponivel_rede, P_temp × f_bal × f_urgencia)
```

**Tempo estimado de carregamento:**
```
SOC_alvo = min(SOC_ideal_max, SOC_necesario + 5%)
delta_E = (SOC_alvo - SOC_actual) / 100 × capacidade_bat
t_carga = delta_E / (P_final × eficiencia) × 60  [minutos]
```

In [0]:
print("=" * 70)
print("FASE 6 — ALOCAÇÃO E AJUSTE DE POTÊNCIA")
print("=" * 70)

# Factor de balanceamento da rede
if P_REDE_ACTUAL > POTENCIA_MAX_REDE * 0.90:
    F_BAL = 0.80
elif P_REDE_ACTUAL > POTENCIA_MAX_REDE * 0.80:
    F_BAL = 0.90
else:
    F_BAL = 1.00

P_DISPONIVEL = POTENCIA_MAX_REDE - P_REDE_ACTUAL

print(f"Factor de balanceamento (f_bal): {F_BAL}")
print(f"Capacidade disponível na rede:   {P_DISPONIVEL} kW")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW alocacoes AS
    WITH autocarros_ord AS (
        SELECT
            device_id,
            nome_rota,
            prioridade,
            soc_actual,
            soc_necesario_pct,
            soc_volta_pct,
            delta_soc,
            temp_bateria,
            score_total,
            paragens_restantes,
            soc_previsto_chegada,
            elegivel,
            razao_elegibilidade,
            ROW_NUMBER() OVER (ORDER BY score_total DESC) AS ordem
        FROM scores_ordenados
        WHERE elegivel = 1
    ),
    estacoes_ord AS (
        SELECT
            device_id AS estacao_id,
            ROW_NUMBER() OVER (ORDER BY device_id) AS ordem
        FROM estacoes_estado
        WHERE disponivel = 1
    ),
    alocados AS (
        SELECT
            a.*,
            e.estacao_id,
            CASE WHEN e.estacao_id IS NOT NULL THEN 1 ELSE 0 END AS tem_estacao
        FROM autocarros_ord a
        LEFT JOIN estacoes_ord e ON a.ordem = e.ordem
    )
    SELECT
        device_id,
        nome_rota,
        prioridade,
        soc_actual,
        delta_soc,
        soc_necesario_pct,
        soc_previsto_chegada,
        temp_bateria,
        score_total,
        paragens_restantes,
        estacao_id,
        tem_estacao,

        -- Potência ajustada por temperatura da bateria (P_temp)
        CASE
            WHEN temp_bateria > {TEMP_BATERIA_MAX}      THEN 0.0
            WHEN temp_bateria > {TEMP_BATERIA_IDEAL_MAX} THEN 80.0
            WHEN temp_bateria < {TEMP_BATERIA_IDEAL_MIN} THEN 50.0
            ELSE {POTENCIA_MAX_ESTACAO}
        END AS p_temp,

        -- Factor de urgência relativa (f_urgencia, só quando Π > 1)
        CASE
            WHEN {PI} > 1 AND score_total >= 150 THEN 1.00
            WHEN {PI} > 1 AND score_total >= 100 THEN 0.90
            WHEN {PI} > 1                        THEN 0.80
            ELSE 1.00
        END AS f_urgencia,

        -- SOC alvo de carregamento
        ROUND(LEAST({SOC_IDEAL_MAX}, soc_necesario_pct + 5), 2) AS soc_alvo_pct,

        elegivel,
        razao_elegibilidade

    FROM alocados
""")

# Calcular potência final e tempo de carregamento
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW alocacoes_final AS
    SELECT
        *,
        -- Potência final = min(P_max, P_disponivel, P_temp × f_bal × f_urgencia)
        ROUND(
            LEAST(
                {POTENCIA_MAX_ESTACAO},
                {P_DISPONIVEL},
                p_temp * {F_BAL} * f_urgencia
            ), 1
        ) AS potencia_final_kw,

        -- Energia a carregar (kWh)
        ROUND(
            GREATEST(soc_alvo_pct - soc_actual, 0) / 100.0 * {BATERIA_CAPACIDADE_KWH},
        2) AS energia_carregar_kwh

    FROM alocacoes
""")

# Tempo estimado de carregamento
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW alocacoes_com_tempo AS
    SELECT
        *,
        -- t_carga = delta_E / (P_final × eficiencia) × 60
        CASE
            WHEN potencia_final_kw > 0
            THEN ROUND(energia_carregar_kwh / (potencia_final_kw * {EFICIENCIA_CARREGAMENTO}) * 60, 1)
            ELSE NULL
        END AS tempo_carga_min
    FROM alocacoes_final
""")

display(spark.sql("""
    SELECT
        device_id,
        nome_rota,
        soc_actual,
        soc_alvo_pct,
        delta_soc,
        estacao_id,
        p_temp,
        f_urgencia,
        potencia_final_kw,
        energia_carregar_kwh,
        tempo_carga_min
    FROM alocacoes_com_tempo
    ORDER BY score_total DESC
"""))

FASE 6 — ALOCAÇÃO E AJUSTE DE POTÊNCIA
Factor de balanceamento (f_bal): 1.0
Capacidade disponível na rede:   120.0 kW


device_id,nome_rota,soc_actual,soc_alvo_pct,delta_soc,estacao_id,p_temp,f_urgencia,potencia_final_kw,energia_carregar_kwh,tempo_carga_min
BUS-001,Bolhao - Castelo do Queijo,17.75,26.17,3.42,null,120.0,1.00,120.0,29.47,16.0


## Fase 7 — Decisões Finais

Tabela consolidada com a decisão e razão detalhada para cada autocarro.

Inclui também os autocarros não elegíveis com a razão da não elegibilidade, e o tempo estimado de libertação das estações ocupadas para os autocarros em fila de espera.

In [0]:
print("=" * 70)
print("FASE 7 — DECISÕES FINAIS")
print("=" * 70)

# Tempo estimado de libertação das estações ocupadas
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW libertacao_estacoes AS
    WITH estacoes_ocupadas AS (
        SELECT
            e.device_id AS estacao_id,
            e.connectedBus,
            e.potencia_kw
        FROM estacoes_estado e
        WHERE e.state = 'occupied' AND e.potencia_kw > 0
    ),
    autocarros_a_carregar AS (
        SELECT
            b.device_id,
            ROUND(b.batteryLevel, 2) AS soc_actual,
            r.soc_necesario_pct
        FROM (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) AS rn
            FROM bus_telemetry_raw
        ) b
        JOIN scores_urgencia r ON b.device_id = r.device_id
        WHERE b.rn = 1
    )
    SELECT
        e.estacao_id,
        e.connectedBus,
        -- Energia ainda a carregar (kWh)
        ROUND(
            GREATEST((a.soc_necesario_pct + 5 - a.soc_actual) / 100.0 * {BATERIA_CAPACIDADE_KWH}, 0),
        2) AS energia_restante_kwh,
        -- Tempo estimado de libertação (minutos)
        ROUND(
            GREATEST((a.soc_necesario_pct + 5 - a.soc_actual) / 100.0 * {BATERIA_CAPACIDADE_KWH}, 0)
            / (e.potencia_kw * {EFICIENCIA_CARREGAMENTO}) * 60,
        1) AS tempo_libertacao_min
    FROM estacoes_ocupadas e
    LEFT JOIN autocarros_a_carregar a ON e.connectedBus = a.device_id
""")

# Decisões finais — autocarros elegíveis
print("\n--- AUTOCARROS ELEGÍVEIS ---")
display(spark.sql(f"""
    SELECT
        device_id                               AS autocarro,
        nome_rota,
        prioridade,
        ROUND(soc_actual, 1)                   AS bateria_actual_pct,
        ROUND(delta_soc, 1)                    AS deficit_pct,
        score_total,
        COALESCE(estacao_id, 'EM FILA')        AS estacao,
        ROUND(potencia_final_kw, 0)            AS potencia_kw,
        ROUND(tempo_carga_min, 0)              AS tempo_estimado_min,
        paragens_restantes,
        ROUND(soc_previsto_chegada, 1)         AS soc_previsto_deposito_pct,
        -- Acção recomendada
        CASE
            WHEN estacao_id IS NOT NULL AND potencia_final_kw > 0
                THEN CONCAT('CARREGAR AGORA em ', estacao_id, ' a ', ROUND(potencia_final_kw,0), ' kW')
            WHEN estacao_id IS NULL
                THEN 'AGUARDAR — sem estação disponível'
            ELSE 'VERIFICAR'
        END AS acao,
        -- Razão detalhada
        CONCAT(
            CASE WHEN score_total >= 160 THEN 'EMERGÊNCIA' WHEN score_total >= 100 THEN 'URGENTE' ELSE 'PRIORITÁRIO' END, ' | ',
            'Défice: ', ROUND(delta_soc,1), '% | ',
            'Score: ', score_total, ' pts | ',
            CASE WHEN soc_previsto_chegada < {SOC_MIN} THEN 'ALERTA: não chega ao depósito em segurança | ' ELSE '' END,
            nome_rota
        ) AS razao
    FROM alocacoes_com_tempo
    ORDER BY score_total DESC
"""))

# --- RESUMO TEXTUAL: AUTOCARROS ELEGÍVEIS ---
rows_el = spark.sql("""
    SELECT device_id, nome_rota, soc_actual, delta_soc, score_total,
           COALESCE(estacao_id, 'EM FILA') AS estacao,
           ROUND(potencia_final_kw, 0) AS potencia_kw,
           ROUND(tempo_carga_min, 0)   AS tempo_min,
           CASE
               WHEN estacao_id IS NOT NULL AND potencia_final_kw > 0
                   THEN CONCAT('CARREGAR AGORA em ', estacao_id, ' a ', ROUND(potencia_final_kw,0), ' kW')
               WHEN estacao_id IS NULL THEN 'AGUARDAR - sem estacao disponivel'
               ELSE 'VERIFICAR'
           END AS acao
    FROM alocacoes_com_tempo ORDER BY score_total DESC
""").collect()
for r in rows_el:
    print(f"  {r['device_id']} | {r['nome_rota']:<30} | SOC={r['soc_actual']:.1f}% | Deficit={r['delta_soc']:.1f}% | Score={r['score_total']} | Estacao={r['estacao']} | {r['potencia_kw']} kW | {r['tempo_min']} min | {r['acao']}")

# Decisões finais — autocarros não elegíveis
print("\n--- AUTOCARROS SEM NECESSIDADE DE CARREGAMENTO ---")
display(spark.sql("""
    SELECT
        device_id        AS autocarro,
        nome_rota,
        ROUND(soc_actual, 1) AS bateria_actual_pct,
        ROUND(delta_soc, 1)  AS deficit_pct,
        razao_elegibilidade  AS estado
    FROM scores_ordenados
    WHERE elegivel = 0
    ORDER BY device_id
"""))

# --- RESUMO TEXTUAL: AUTOCARROS SEM NECESSIDADE ---
rows_ne = spark.sql("""
    SELECT device_id, nome_rota, ROUND(soc_actual,1) AS soc_actual, ROUND(delta_soc,1) AS delta_soc, razao_elegibilidade
    FROM scores_ordenados WHERE elegivel = 0 ORDER BY device_id
""").collect()
for r in rows_ne:
    print(f"  {r['device_id']} | {r['nome_rota']:<30} | SOC={r['soc_actual']}% | {r['razao_elegibilidade']}")

# Tempo de libertação das estações ocupadas
print("\n--- TEMPO ESTIMADO DE LIBERTAÇÃO DAS ESTAÇÕES OCUPADAS ---")
display(spark.sql("""
    SELECT
        estacao_id,
        connectedBus        AS autocarro_ligado,
        energia_restante_kwh,
        tempo_libertacao_min
    FROM libertacao_estacoes
    ORDER BY tempo_libertacao_min
"""))

# --- RESUMO TEXTUAL: LIBERTACAO DE ESTACOES OCUPADAS ---
rows_lib = spark.sql("""
    SELECT estacao_id, connectedBus AS autocarro_ligado, energia_restante_kwh, tempo_libertacao_min
    FROM libertacao_estacoes ORDER BY tempo_libertacao_min
""").collect()
for r in rows_lib:
    print(f"  {r['estacao_id']} | Ligado: {r['autocarro_ligado']} | Energia restante: {r['energia_restante_kwh']} kWh | Libertacao em: {r['tempo_libertacao_min']} min")
if not rows_lib:
    print("  Nenhuma estacao ocupada neste momento.")

print("\n" + "=" * 70)
print("ALGORITMO CONCLUÍDO")
print(f"Indicador de pressão (Π): {PI}")
print(f"Factor de balanceamento:  {F_BAL}")
print("=" * 70)

FASE 7 — DECISÕES FINAIS

--- AUTOCARROS ELEGÍVEIS ---


autocarro,nome_rota,prioridade,bateria_actual_pct,deficit_pct,score_total,estacao,potencia_kw,tempo_estimado_min,paragens_restantes,soc_previsto_deposito_pct,acao,razao
BUS-001,Bolhao - Castelo do Queijo,Alta,17.8,3.4,90,EM FILA,120,16.0,2.0,16.6,AGUARDAR — sem estação disponível,PRIORITÁRIO | Défice: 3.4% | Score: 90 pts | ALERTA: não chega ao depósito em segurança | Bolhao - Castelo do Queijo


  BUS-001 | Bolhao - Castelo do Queijo     | SOC=17.8% | Deficit=3.4% | Score=90 | Estacao=EM FILA | 120 kW | 16.0 min | AGUARDAR - sem estacao disponivel

--- AUTOCARROS SEM NECESSIDADE DE CARREGAMENTO ---


autocarro,nome_rota,bateria_actual_pct,deficit_pct,estado
BUS-002,Aliados - Viso,45.9,-25.9,SEM NECESSIDADE - Bateria suficiente para completar o percurso
BUS-003,Aliados - Passeio Alegre,81.2,-58.0,SEM NECESSIDADE - Bateria suficiente para completar o percurso


  BUS-002 | Aliados - Viso                 | SOC=45.9% | SEM NECESSIDADE - Bateria suficiente para completar o percurso
  BUS-003 | Aliados - Passeio Alegre       | SOC=81.2% | SEM NECESSIDADE - Bateria suficiente para completar o percurso

--- TEMPO ESTIMADO DE LIBERTAÇÃO DAS ESTAÇÕES OCUPADAS ---


estacao_id,autocarro_ligado,energia_restante_kwh,tempo_libertacao_min
CS-001,BUS-002,0.0,0.0
CS-003,BUS-003,0.0,0.0


  CS-001 | Ligado: BUS-002 | Energia restante: 0.0 kWh | Libertacao em: 0.0 min
  CS-003 | Ligado: BUS-003 | Energia restante: 0.0 kWh | Libertacao em: 0.0 min

ALGORITMO CONCLUÍDO
Indicador de pressão (Π): 1.0
Factor de balanceamento:  1.0


### Persistência para consumo externo (dashboard)

As fases anteriores trabalham sobre `alocacoes_com_tempo`, uma vista temporária válida apenas dentro desta sessão do notebook. Como o dashboard Lakeview usa um SQL Warehouse separado (sem acesso a essa vista), esta célula grava o mesmo resultado numa tabela Delta persistente, que os datasets do dashboard passam a consultar diretamente.

In [0]:
# Persiste o resultado da Fase 7 numa tabela Delta, para poder ser lido
# por computacao separada (ex. SQL Warehouse de um Lakeview Dashboard),
# que nao tem acesso as vistas temporarias desta sessao do notebook.
spark.sql("""
    CREATE OR REPLACE TABLE decisoes_carregamento AS
    SELECT * FROM alocacoes_com_tempo
""")

print("Tabela decisoes_carregamento atualizada.")

Tabela decisoes_carregamento atualizada.


## Fase 8 — Registo de Alertas Operacionais

Fase complementar às sete fases centrais do algoritmo de decisão, que traduz os
resultados das Fases 3 e 7 em alertas persistentes, gravados na tabela Delta
`alertas_operacionais`. Em vez de a informação crítica estar apenas disponível para
quem consulta ativamente a interface, fica registada de forma durável e consultável —
por exemplo por um Databricks Workflow ou Azure Logic App que leia esta tabela e
dispare uma notificação real (email, SMS ou push).

Quatro tipos de alerta são gerados, cada um com uma severidade (CRITICA, ALTA ou MEDIA):

- **BATERIA_CRITICA** — autocarro elegível para carregamento (Fase 3), classificado pelo score total.
- **SEM_ESTACAO_DISPONIVEL** — autocarro elegível sem estação atribuída (em fila de espera).
- **RISCO_NAO_REGRESSO** — SOC previsto à chegada ao depósito abaixo do SOC_MIN.
- **ESTACAO_AVARIA** — estação de carregamento em estado fault.


In [0]:
print("=" * 70)
print("FASE 8 — REGISTO DE ALERTAS OPERACIONAIS")
print("=" * 70)

from pyspark.sql.functions import current_timestamp, lit

spark.sql("""
    CREATE TABLE IF NOT EXISTS alertas_operacionais (
        gerado_em TIMESTAMP,
        device_id STRING,
        tipo_alerta STRING,
        severidade STRING,
        mensagem STRING
    ) USING DELTA
""")

# BATERIA_CRITICA + SEM_ESTACAO_DISPONIVEL + RISCO_NAO_REGRESSO,
# derivados das vistas ja criadas pela Fase 7 (alocacoes_com_tempo)
alertas_autocarros = spark.sql(f"""
    SELECT
        current_timestamp() as gerado_em,
        device_id,
        'BATERIA_CRITICA' as tipo_alerta,
        CASE WHEN score_total >= 160 THEN 'CRITICA'
             WHEN score_total >= 100 THEN 'ALTA'
             ELSE 'MEDIA' END as severidade,
        CONCAT(
            device_id, ' (', nome_rota, '): bateria a ', ROUND(soc_actual,1),
            '%, deficit de ', ROUND(delta_soc,1), '%, score ', score_total,
            '. Nao deve iniciar nova rota sem carregar primeiro.'
        ) as mensagem
    FROM alocacoes_com_tempo
    WHERE elegivel = 1

    UNION ALL

    SELECT
        current_timestamp() as gerado_em,
        device_id,
        'SEM_ESTACAO_DISPONIVEL' as tipo_alerta,
        'ALTA' as severidade,
        CONCAT(
            device_id, ' (', nome_rota, '): elegível para carregamento (score ', score_total,
            ') mas sem estacao disponivel - em fila de espera.'
        ) as mensagem
    FROM alocacoes_com_tempo
    WHERE elegivel = 1 AND estacao_id IS NULL

    UNION ALL

    SELECT
        current_timestamp() as gerado_em,
        device_id,
        'RISCO_NAO_REGRESSO' as tipo_alerta,
        'CRITICA' as severidade,
        CONCAT(
            device_id, ' (', nome_rota, '): SOC previsto a chegada ao deposito (',
            ROUND(soc_previsto_chegada,1), '%) abaixo do minimo de seguranca (',
            {SOC_MIN}, '%).'
        ) as mensagem
    FROM alocacoes_com_tempo
    WHERE soc_previsto_chegada < {SOC_MIN}
""")

# ESTACAO_AVARIA, derivado da vista estacoes_estado (Fase 3)
alertas_estacoes = spark.sql("""
    SELECT
        current_timestamp() as gerado_em,
        device_id,
        'ESTACAO_AVARIA' as tipo_alerta,
        'ALTA' as severidade,
        CONCAT(device_id, ': estacao em avaria, indisponivel para alocacao.') as mensagem
    FROM estacoes_estado
    WHERE state = 'fault'
""")

novos_alertas = alertas_autocarros.unionAll(alertas_estacoes)
novos_alertas.write.mode("append").saveAsTable("alertas_operacionais")

n_novos = novos_alertas.count()
print(f"\n{n_novos} alerta(s) gerado(s) e gravado(s) em alertas_operacionais.")

print("\n--- ALERTAS DESTA EXECUCAO ---")
display(novos_alertas.orderBy(
    novos_alertas.severidade.desc()
))

print("\n--- ALERTAS ATIVOS (ultimos 10 minutos, por severidade) ---")
display(spark.sql("""
    SELECT gerado_em, device_id, tipo_alerta, severidade, mensagem
    FROM alertas_operacionais
    WHERE gerado_em >= current_timestamp() - INTERVAL 10 MINUTES
    ORDER BY
        CASE severidade WHEN 'CRITICA' THEN 1 WHEN 'ALTA' THEN 2 ELSE 3 END,
        gerado_em DESC
"""))


FASE 8 — REGISTO DE ALERTAS OPERACIONAIS

4 alerta(s) gerado(s) e gravado(s) em alertas_operacionais.

--- ALERTAS DESTA EXECUCAO ---


gerado_em,device_id,tipo_alerta,severidade,mensagem
2026-10-17T19:46:45.857Z,BUS-001,BATERIA_CRITICA,MEDIA,"BUS-001 (Bolhao - Castelo do Queijo): bateria a 17.8%, deficit de 3.4%, score 90. Nao deve iniciar nova rota sem carregar primeiro."
2026-10-17T19:46:45.857Z,BUS-001,RISCO_NAO_REGRESSO,CRITICA,BUS-001 (Bolhao - Castelo do Queijo): SOC previsto a chegada ao deposito (16.6%) abaixo do minimo de seguranca (20.0%).
2026-10-17T19:46:45.857Z,BUS-001,SEM_ESTACAO_DISPONIVEL,ALTA,BUS-001 (Bolhao - Castelo do Queijo): elegível para carregamento (score 90) mas sem estacao disponivel - em fila de espera.
2026-10-17T19:46:45.857Z,CS-002,ESTACAO_AVARIA,ALTA,"CS-002: estacao em avaria, indisponivel para alocacao."



--- ALERTAS ATIVOS (ultimos 10 minutos, por severidade) ---


gerado_em,device_id,tipo_alerta,severidade,mensagem
2026-10-17T19:46:37.165Z,BUS-001,RISCO_NAO_REGRESSO,CRITICA,BUS-001 (Bolhao - Castelo do Queijo): SOC previsto a chegada ao deposito (16.6%) abaixo do minimo de seguranca (20.0%).
2026-10-17T19:46:37.165Z,CS-002,ESTACAO_AVARIA,ALTA,"CS-002: estacao em avaria, indisponivel para alocacao."
2026-10-17T19:46:37.165Z,BUS-001,SEM_ESTACAO_DISPONIVEL,ALTA,BUS-001 (Bolhao - Castelo do Queijo): elegível para carregamento (score 90) mas sem estacao disponivel - em fila de espera.
2026-10-17T19:46:37.165Z,BUS-001,BATERIA_CRITICA,MEDIA,"BUS-001 (Bolhao - Castelo do Queijo): bateria a 17.8%, deficit de 3.4%, score 90. Nao deve iniciar nova rota sem carregar primeiro."
